<h1>🧬 Biofilter — Report: <code>annotation_master_gene</code></h1>

Everything the bundle knows about a list of genes, one row per input:
canonical IDs, HGNC metadata, build 38 coordinates, relationship counts
by related entity group, and the number of variants inside the gene's
range.

Reads the bundle natively (ADR-004). Accepts symbols, aliases, synonyms
or cross-reference codes, matched case-insensitively.

### 1. Open a bundle

In [2]:
from biofilter import Biofilter

BUNDLE = "/Users/andrerico/Works/Sys/biofilter_430/biofilter_data/bundles/20260914" # change this to the path of your biofilter_data bundle
REPORT = "annotation_master_gene"

bf = Biofilter(bundle=BUNDLE, debug_mode=False)
bf

[INFO] ════════════════════════════════════
[INFO] 🚀 Initializing Biofilter
[INFO]    • Version: 4.3.0
[INFO]    • Debug mode: False
[INFO]    • Config: /Users/andrerico/Works/Sys/biofilter_430/.biofilter.toml
[INFO]    • DB URI: parquet:///Users/andrerico/Works/Sys/biofilter_430/biofilter_data/bundles/20260914
[INFO] ════════════════════════════════════
[INFO] 🔌 Database connection established
[INFO]    • Engine: duckdb+parquet
[INFO]    • Host:   parquet bundle
[INFO]    • DB:     /Users/andrerico/Works/Sys/biofilter_430/biofilter_data/bundles/20260914/tables
[INFO]    • Views:  37 (read-only)
[INFO]    • Time:   142.7 ms
[INFO] ════════════════════════════════════


<Biofilter(db_uri=parquet:///Users/andrerico/Works/Sys/biofilter_430/biofilter_data/bundles/20260914)>

### 2. What the report offers

In [3]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

columns:
  input_value
  input_matched_alias
  entity_id
  gene_symbol
  hgnc_id
  ensembl_id
  entrez_id
  hgnc_status
  omic_status
  gene_locus_group
  gene_locus_type
  gene_groups
  build
  chromosome
  start_position
  end_position
  entity_relationships_by_group
  total_entity_relationships
  variant_count_in_gene_range
  other_aliases
  status
  note

example input:
{'input_data': ['TP53', 'BRCA1', 'EGFR'], 'include_relationships': True, 'include_variant_summary': True, 'emit_not_found_rows': True}


In [4]:
print(bf.report.explain(REPORT))

# annotation_master_gene

Everything the bundle knows about a list of genes, one row per input.

```bash
biofilter --bundle /path/to/bundle report run \
    --report-name annotation_master_gene \
    --input TP53 --input BRCA1 \
    --output genes.csv
```

The result is written as CSV with a `genes.csv.provenance.json` beside
it, naming the bundle the rows came from. That matters here because
`entity_id` is scoped to one build: the same integer means a different
gene in the next bundle.

## Input

Gene symbols, aliases, synonyms, or cross-reference codes (`HGNC:11998`,
`ENSG00000141510`, `7157`). Matching is case-insensitive.

`--input __ALL__` annotates every gene entity in the bundle instead.

## Parameters

| parameter | default | meaning |
| --- | --- | --- |
| `input_data` | required | genes, via `--input`/`--input-file`, or `__ALL__` |
| `include_relationships` | `true` | count relationships by related entity group |
| `include_variant_summary` | `true` | count variants inside th

### 3. Run it

Any of these resolve to the same gene — symbol, synonym, or code:

```
TP53   p53   HGNC:11998   ENSG00000141510   7157
```

In [5]:
input_genes = [
    "TP53",
    "BRCA1",
    "ENSG00000146648",   # EGFR, by Ensembl id
    "NOT_A_GENE",        # kept in the output, with status='not_found'
]

result = bf.report.run(
    REPORT,
    input_data=input_genes,
    include_relationships=True,
    include_variant_summary=True,
    emit_not_found_rows=True,
)

df = result.to_pandas()
print(f"{result.num_rows} rows from bundle {result.provenance['bundle_id']}")
df[["input_value", "input_matched_alias", "gene_symbol", "entity_id", "status"]]

[INFO] Report 'annotation_master_gene' produced 4 rows in 0.84s from bundle f47a1c47f3ac95f3.


4 rows from bundle f47a1c47f3ac95f3


,input_value,input_matched_alias,gene_symbol,entity_id,status
0,BRCA1,BRCA1,BRCA1,22857.0,ok
1,ENSG00000146648,ENSG00000146648,EGFR,46168.0,ok
2,NOT_A_GENE,None,None,NaN,not_found
3,TP53,TP53,TP53,37360.0,ok


### 4. Reading the result

`status` is the first column to look at.

| value | meaning |
| --- | --- |
| `ok` | resolved, with a gene record and build 38 coordinates |
| `partial` | resolved, but something is missing — `note` says what |
| `not_found` | the bundle has no gene entity for this input |

Unresolved inputs are **kept on purpose**. Dropping them would leave no
way to tell "absent from this bundle" from "never asked for".

In [6]:
df[["input_value", "status", "note"]]

,input_value,status,note
0,BRCA1,ok,None
1,ENSG00000146648,ok,None
2,NOT_A_GENE,not_found,Input not resolved to a Gene entity.
3,TP53,ok,None


#### Identity and coordinates

In [7]:
df[[
    "input_value",
    "gene_symbol",
    "hgnc_id",
    "ensembl_id",
    "entrez_id",
    "hgnc_status",
    "omic_status",
    "gene_locus_group",
    "build",
    "chromosome",
    "start_position",
    "end_position",
]]

,input_value,gene_symbol,hgnc_id,ensembl_id,entrez_id,hgnc_status,omic_status,gene_locus_group,build,chromosome,start_position,end_position
0,BRCA1,BRCA1,HGNC:1100,ENSG00000012048,672,Approved,active,protein-coding gene,38.0,17.0,43044292.0,43170245.0
1,ENSG00000146648,EGFR,HGNC:3236,ENSG00000146648,1956,Approved,active,protein-coding gene,38.0,7.0,55018820.0,55211628.0
2,NOT_A_GENE,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN
3,TP53,TP53,HGNC:11998,ENSG00000141510,7157,Approved,active,protein-coding gene,38.0,17.0,7661779.0,7687546.0


#### Lists: groups, relationships, other aliases

Three columns hold lists rather than scalars. In a DataFrame and in
parquet they are real lists; exported to CSV they are written as JSON so
one cell can hold them.

In [8]:
for _, row in df[df["status"] != "not_found"].iterrows():
    print(row["gene_symbol"])
    print("  gene groups :", list(row["gene_groups"]))
    print("  relationships:", row["total_entity_relationships"], "total")
    for entry in row["entity_relationships_by_group"]:
        print(f"      {entry['group_name']:<12} {entry['count']:>6}")
    print("  other aliases:", list(row["other_aliases"])[:6])
    print()

BRCA1
  gene groups : ['BRCA1 A complex', 'BRCA1 B complex', 'BRCA1 C complex', 'FA complementation groups', 'Protein phosphatase 1 regulatory subunits', 'Ring finger proteins']
  relationships: 2551 total
      Genes          1242
      Proteins       1238
      Pathways         67
      Diseases          4
  other aliases: ['BRCA1 DNA repair associated', 'BRCA1/BRCA2-containing complex, subunit 1', 'BRCC1', 'FANCS', 'Fanconi anemia, complementation group S', 'PPP1R53']

EGFR
  gene groups : ['Erb-b2 receptor tyrosine kinases']
  relationships: 6437 total
      Genes          3178
      Proteins       3131
      Pathways        126
      Diseases          2
  other aliases: ['ERBB', 'ERBB1', 'ERRP', 'epidermal growth factor receptor', 'epidermal growth factor receptor (avian erythroblastic leukemia viral (v-erb-b) oncogene homolog)', 'erb-b2 receptor tyrosine kinase 1']

TP53
  gene groups : ['p53 transcription factor family']
  relationships: 5181 total
      Genes          2511
    

Relationships are counted **in both directions**: a gene appearing on
either side of a relationship counts it, grouped by what is on the other
side.

#### `variant_count_in_gene_range`: null is not zero

| value | meaning |
| --- | --- |
| a number | the range was searched, and held that many variants |
| `0` | the range was searched and held none |
| `NaN` / null | the count was **not made** |

Null happens when the gene has no build 38 range, or when
`include_variant_summary=False`.

⚠️ A bundle built for a subset of chromosomes returns `0` for every gene
outside them. That is true of the bundle, not of the genome — check what
the bundle covers before reading a zero as biology.

In [9]:
df[["input_value", "chromosome", "start_position", "end_position",
    "variant_count_in_gene_range"]]

,input_value,chromosome,start_position,end_position,variant_count_in_gene_range
0,BRCA1,17.0,43044292.0,43170245.0,0.0
1,ENSG00000146648,7.0,55018820.0,55211628.0,0.0
2,NOT_A_GENE,NaN,NaN,NaN,NaN
3,TP53,17.0,7661779.0,7687546.0,0.0


### 5. Lighter modes

`include_relationships` and `include_variant_summary` are the two
expensive sections. Turning them off is what makes whole-catalog mode
comfortable.

In [10]:
fast = bf.report.run(
    REPORT,
    input_data=input_genes,
    include_relationships=False,
    include_variant_summary=False,
)

fast.to_pandas()[["input_value", "gene_symbol", "hgnc_id", "chromosome", "status"]]

[INFO] Report 'annotation_master_gene' produced 4 rows in 0.10s from bundle f47a1c47f3ac95f3.


,input_value,gene_symbol,hgnc_id,chromosome,status
0,BRCA1,BRCA1,HGNC:1100,17.0,ok
1,ENSG00000146648,EGFR,HGNC:3236,7.0,ok
2,NOT_A_GENE,None,None,NaN,not_found
3,TP53,TP53,HGNC:11998,17.0,ok


### 6. Every gene in the bundle

`input_data="__ALL__"` annotates every gene entity instead of a list.

In [11]:
import time

started = time.perf_counter()
everything = bf.report.run(REPORT, input_data="__ALL__")
elapsed = time.perf_counter() - started

catalog = everything.to_pandas()
print(f"{everything.num_rows:,} genes in {elapsed:.1f}s")
print(catalog["status"].value_counts().to_dict())

[INFO] Report 'annotation_master_gene' produced 72,660 rows in 1.01s from bundle f47a1c47f3ac95f3.


72,660 genes in 1.0s
{'ok': 39306, 'partial': 33354}


In [12]:
# Genes with no build 38 location are the 'partial' ones, and they are
# also exactly the rows whose variant count is null.
partial = catalog[catalog["status"] == "partial"]
print(f"{len(partial):,} without a build 38 location")
print(f"{catalog['variant_count_in_gene_range'].isna().sum():,} with a null variant count")

33,354 without a build 38 location
33,354 with a null variant count


In [13]:
# What the bundle actually covers, which is what a zero above means.
covered = catalog.dropna(subset=["variant_count_in_gene_range"])
covered.groupby("chromosome")["variant_count_in_gene_range"].agg(
    genes="size", with_variants=lambda s: int((s > 0).sum())
).sort_values("with_variants", ascending=False).head(10)

,genes,with_variants
chromosome,,
22.0,962,958
2.0,2914,0
21.0,559,0
20.0,1059,0
19.0,2155,0
18.0,694,0
17.0,1982,0
16.0,1511,0
15.0,1373,0


### 7. Export

CSV is the default. The `.provenance.json` written beside it records
which bundle the ids came from — necessary, because `entity_id` means a
different gene in the next build.

In [14]:
for path in everything.write("annotation_master_gene.csv"):
    print(path)

annotation_master_gene.csv
annotation_master_gene.csv.provenance.json


In [ ]:
# Parquet keeps the list columns as lists, and carries the provenance in
# the file's own metadata.
everything.write("annotation_master_gene.parquet")

### 8. The same thing on the command line

```bash
biofilter --bundle /path/to/bundles/20260914 report run \
    --report-name annotation_master_gene \
    --input TP53 --input BRCA1 \
    --param include_variant_summary=false \
    --output genes.csv
```

`--input-file genes.txt` takes one value per line.

### 9. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("unresolved inputs:", int((df["status"] == "not_found").sum()))
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))